In [18]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import os

In [19]:
spark = SparkSession.builder.appName("Office_Products_Cleaning").getOrCreate()

In [20]:
df = spark.read.format("json").option("inferSchema","true").load("../data/raw/Office_Products.jsonl")

In [21]:
initial_count = df.count()

In [22]:
df.show(5)


+----------+------------+--------------------+-----------+------+--------------------+-------------+--------------------+--------------------+-----------------+
|      asin|helpful_vote|              images|parent_asin|rating|                text|    timestamp|               title|             user_id|verified_purchase|
+----------+------------+--------------------+-----------+------+--------------------+-------------+--------------------+--------------------+-----------------+
|B01AHHL4X2|           0|                  []| B01MZ3SD2X|   5.0|Lovely ink. Write...|1677939345945| Pretty & I love it!|AFKZENTNBQ7A7V7UX...|             true|
|B08L6H23JZ|           0|                  []| B08L6H23JZ|   4.0|Overall I’m prett...|1677939160682|2 excellent 1 ext...|AFKZENTNBQ7A7V7UX...|             true|
|B07JDZ5J46|           2|                  []| B07JDZ5J46|   1.0|[[VIDEOID:63276c1...|1660188831933|I don’t get the r...|AFKZENTNBQ7A7V7UX...|             true|
|B004MNX7EW|           0|[{IMAGE, 

In [23]:
df = df.withColumn("review_text", concat_ws(" ", col("title"),col("text")))

In [24]:
df = df.filter(col("review_text").isNotNull())
df = df.filter(trim(col("review_text")) != "")

In [25]:
df = df.dropDuplicates(["review_text","user_id","parent_asin"])

In [26]:
after_clean_count = df.count()

26/03/29 21:12:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


In [27]:
df = df.withColumn("review_text", regexp_replace(col("review_text"), "[^a-zA-Z0-9\\.\\?\\!\\s]", ""))
df = df.withColumn("review_text", regexp_replace(col("review_text"),"\\s+"," "))
df = df.withColumn("review_text", trim(col("review_text")))

In [28]:
df = df.withColumn("review_id", monotonically_increasing_id()+1)

In [29]:
df.select("review_text").show(5, truncate=False)

26/03/29 21:13:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|review_text                                                                                                                                                                                                                                                                                             |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Its beautiful. I buy calendars for the whole family Its beautiful. I buy calendars for the whole famil

In [30]:
df = df.withColumn("sentence_text", explode(split(col("review_text"), "[\\.\\!\\?]+")))
df = df.withColumn(
    "sentence_text",
    regexp_replace(col("sentence_text"), "\\bbr\\b", "")
)
df = df.withColumn(
    "sentence_text",
    regexp_replace(col("sentence_text"), "^\\s+", "")
)

df = df.withColumn(
    "sentence_text",
    regexp_replace(col("sentence_text"), "\\s+$", "")
)

df = df.withColumn(
    "sentence_text",
    regexp_replace(col("sentence_text"), "\\s+", " ")
)
df = df.filter(col("sentence_text") != "")

In [31]:
df = df.withColumn("sentence_id", row_number().over(Window.partitionBy("review_id").orderBy("sentence_text")))

In [32]:
df_final = df.select(
    "parent_asin",
    "review_id",
    "sentence_id",
    "sentence_text",
    "rating"
)

In [33]:
df_final.show(10, truncate=False)

26/03/29 21:14:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/29 21:14:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/29 21:14:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/29 21:14:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/29 21:14:40 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/29 21:14:40 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/29 21:14:41 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/29 21:14:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/29 21:14:49 WARN RowBasedKeyValueBatch: Calling spill() on

+-----------+---------+-----------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+
|parent_asin|review_id|sentence_id|sentence_text                                                                                                                                                                                                |rating|
+-----------+---------+-----------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+
|B083H1KV7Z |19       |1          |Great quality                                                                                                                                                                                                |5.0   |
|B08

In [34]:
sentence_count = df_final.count()

26/03/29 21:18:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


In [35]:
output_path = "../data/processed/Office_Products_Cleaned.parquet"

In [36]:
df_final.write.mode("overwrite").parquet(output_path)


26/03/29 21:21:03 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/29 21:21:03 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/29 21:21:03 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/29 21:21:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/29 21:21:09 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/29 21:21:09 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/29 21:21:09 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/29 21:21:16 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/29 21:21:20 WARN RowBasedKeyValueBatch: Calling spill() on

In [41]:
def get_size(path):
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        print(f"Directory: {dirpath}, Subdirectories: {dirnames}, Files: {filenames}")
        for f in filenames:
           
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total

json_size = get_size("../data/raw")
parquet_size = get_size("../data/processed")

Directory: ../data/raw, Subdirectories: [], Files: ['example_data.jsonl', 'Office_Products.jsonl']
Directory: ../data/processed, Subdirectories: ['Office_Products_Cleaned.parquet'], Files: []
Directory: ../data/processed/Office_Products_Cleaned.parquet, Subdirectories: [], Files: ['.part-00000-e414678c-611d-4b3d-9156-90938292daf5-c000.snappy.parquet.crc', '.part-00001-e414678c-611d-4b3d-9156-90938292daf5-c000.snappy.parquet.crc', '.part-00002-e414678c-611d-4b3d-9156-90938292daf5-c000.snappy.parquet.crc', '.part-00003-e414678c-611d-4b3d-9156-90938292daf5-c000.snappy.parquet.crc', '.part-00004-e414678c-611d-4b3d-9156-90938292daf5-c000.snappy.parquet.crc', '.part-00005-e414678c-611d-4b3d-9156-90938292daf5-c000.snappy.parquet.crc', '.part-00006-e414678c-611d-4b3d-9156-90938292daf5-c000.snappy.parquet.crc', '.part-00007-e414678c-611d-4b3d-9156-90938292daf5-c000.snappy.parquet.crc', '.part-00008-e414678c-611d-4b3d-9156-90938292daf5-c000.snappy.parquet.crc', '.part-00009-e414678c-611d-4b3d-91

In [43]:
report_path = "../outputs/reports/reviews_cleaning_report_office_products.txt"

with open(report_path, "w", encoding="utf-8") as f:
    f.write(f"Số review ban đầu: {initial_count}\n")
    f.write(f"Số review còn lại: {after_clean_count}\n")
    f.write(f"Số câu sau khi tách: {sentence_count}\n")
    f.write(f"Kích thước ban đầu: {json_size / (1024*1024):.2f} MB\n")
    f.write(f"Kích thước cuối cùng: {parquet_size / (1024*1024):.2f} MB\n")